# Otvoreni tok: predvidi → izračunaj → provjeri

Za pravokutni kanal specifična energija po jedinici širine glasi

\[
E(y)=y+\frac{q^2}{2gy^2}.
\]

Ista energija iznad minimuma ima dvije alternativne dubine, ali rubni uvjeti odlučuju koja se grana ostvaruje. Hidraulički skok povezuje nadkritični i podkritični tok uz disipaciju energije.

## Predvidi

1. Skiciraj \(E(y)\), označi kritičnu dubinu i dvije grane.
2. Koja alternativna dubina ima \(Fr>1\)?
3. Kako nesigurnost mjerenja uzvodne dubine i brzine prelazi na procijenjenu spregnutu dubinu skoka?

4. Kako povišenje dna mijenja podkritičnu dubinu i raspoloživu energiju?
5. Zašto isti izmjereni protok nije neovisan podatak na dvama presjecima skoka?
6. Može li čišćenje kanala povećati protok iznad mogućnosti nizvodnog bazena?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
g, q = 9.81, 1.5

def energy(y, discharge_per_width=q):
    return y + discharge_per_width**2/(2*g*y**2)

def froude(y, discharge_per_width=q):
    return discharge_per_width/(y*np.sqrt(g*y))

def bisect_target(function, a, b, target, tolerance=1e-12, max_iter=100):
    fa, fb = function(a)-target, function(b)-target
    if fa*fb > 0:
        raise ValueError("Interval ne omeđuje korijen.")
    history = []
    for iteration in range(max_iter):
        m = 0.5*(a+b)
        fm = function(m)-target
        history.append((iteration, m, fm))
        if abs(fm) < tolerance:
            return m, np.asarray(history)
        if fa*fm <= 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    raise RuntimeError("Bisekcija nije konvergirala.")

yc = (q**2/g)**(1/3)
Emin = 1.5*yc
E_target = 1.20
y_shallow, hist_shallow = bisect_target(energy, 0.05, yc, E_target)
y_deep, hist_deep = bisect_target(energy, yc, 4.0, E_target)
print(f"yc={yc:.4f} m; Emin={Emin:.4f} m")
print(f"Alternativne dubine: {y_shallow:.4f} m (Fr={froude(y_shallow):.3f}) i {y_deep:.4f} m (Fr={froude(y_deep):.3f})")


## Izračunaj: hidraulički skok i mjerna nesigurnost

Za pravokutni kanal spregnuta dubina iz uzvodne dubine \(y_1\) i brzine \(v_1\) jest

\[
y_2=\frac{y_1}{2}\left(\sqrt{1+8Fr_1^2}-1\right).
\]

Monte Carlo uzorkovanje uspoređujemo s lokalnom linearizacijom dobivenom centriranim razlikama. Time razlikujemo nesigurnost ulaza od fizikalnog gubitka energije kroz skok.


In [ ]:
def conjugate_depth(y1, v1):
    Fr1 = v1/np.sqrt(g*y1)
    return 0.5*y1*(np.sqrt(1+8*Fr1**2)-1)

y1, v1 = 0.250, 6.00
y2 = conjugate_depth(y1, v1)
q_jump = y1*v1
Fr1 = v1/np.sqrt(g*y1)
energy_loss = (y2-y1)**3/(4*y1*y2)
momentum_1 = y1**2/2 + q_jump**2/(g*y1)
momentum_2 = y2**2/2 + q_jump**2/(g*y2)

sigma_y, sigma_v = 0.003, 0.08
dy = sigma_y*1e-3
dv = sigma_v*1e-3
grad_y = (conjugate_depth(y1+dy, v1)-conjugate_depth(y1-dy, v1))/(2*dy)
grad_v = (conjugate_depth(y1, v1+dv)-conjugate_depth(y1, v1-dv))/(2*dv)
u_y2_linear = np.sqrt((grad_y*sigma_y)**2 + (grad_v*sigma_v)**2)

rng = np.random.default_rng(20260804)
n_samples = 40_000
y1_mc = rng.normal(y1, sigma_y, n_samples)
v1_mc = rng.normal(v1, sigma_v, n_samples)
y2_mc = conjugate_depth(y1_mc, v1_mc)
u_y2_mc = np.std(y2_mc, ddof=1)
interval = np.quantile(y2_mc, [0.025, 0.975])

print(f"Skok: Fr1={Fr1:.3f}, y2={y2:.4f} m, ΔE={energy_loss:.4f} m")
print(f"u(y2) linearno={u_y2_linear:.5f} m; Monte Carlo={u_y2_mc:.5f} m")
print(f"95 %-tni interval y2=[{interval[0]:.4f}, {interval[1]:.4f}] m")


## Provjeri

Alternativne dubine moraju vratiti istu specifičnu energiju i ležati s različitih strana kritičnog stanja. Za skok provjeravamo očuvanje funkcije količine gibanja te slaganje dvaju postupaka propagacije nesigurnosti.


In [ ]:
assert np.isclose(energy(y_shallow), E_target, rtol=1e-11)
assert np.isclose(energy(y_deep), E_target, rtol=1e-11)
assert froude(y_shallow) > 1 and froude(y_deep) < 1
assert np.isclose(momentum_1, momentum_2, rtol=1e-12)
assert y2 > y1 and energy_loss > 0
assert abs(u_y2_mc/u_y2_linear-1) < 0.05

ys = np.linspace(0.12, 2.0, 500)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ys, energy(ys), color="#256d85", lw=2)
axes[0].axhline(E_target, color="#7a8a96", ls="--")
axes[0].plot([y_shallow, y_deep], [E_target, E_target], "o", color="#b43c35")
axes[0].plot(yc, Emin, "s", color="#2d7d46", label="kritično stanje")
axes[0].set(xlabel="dubina y (m)", ylabel="specifična energija E (m)", title="Dvije grane iste energije")
axes[0].legend()
axes[1].hist(y2_mc, bins=55, color="#7cb5d6", edgecolor="white")
axes[1].axvline(y2, color="#b43c35", lw=2, label="nominalno")
axes[1].set(xlabel="spregnuta dubina y2 (m)", ylabel="broj uzoraka", title="Nesigurnost procjene skoka")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Z4: kontrolni presjek na povišenom dnu

Sada je \(q=2{,}20\ \mathrm{m^2/s}\), uzvodna dubina \(1{,}200\ \mathrm{m}\),
a povišenje dna \(0{,}120\ \mathrm{m}\). Zadržavamo energijsku bilancu i
podkritičnu granu uz izostanak gubitaka. Granica prolaza nastaje kada
raspoloživa specifična energija dosegne minimum. Preko granice zadani
protok i uzvodna dubina više nisu spojivi; korijen ne treba silom tražiti.

In [ ]:
q_hump, y_up, dz = 2.20, 1.200, 0.120
yc_hump = (q_hump**2/g)**(1/3)
E_up = energy(y_up, q_hump)
E_min_hump = 1.5*yc_hump
dz_max = E_up-E_min_hump
def hump_roots(height):
    target = E_up-height
    if target < E_min_hump-1e-12:
        raise ValueError("Prag traži uzvodni uspor ili promjenu protoka.")
    if np.isclose(target,E_min_hump,rtol=0,atol=1e-12):
        return yc_hump,yc_hump
    eq = lambda y: energy(y,q_hump)
    shallow,_ = bisect_target(eq,.01,yc_hump,target)
    deep,_ = bisect_target(eq,yc_hump,2*y_up,target)
    return shallow,deep
shallow_hump,deep_hump = hump_roots(dz)
Fr_crest = froude(deep_hump,q_hump)
assert abs(dz+energy(deep_hump,q_hump)-E_up)<1e-6
assert shallow_hump<yc_hump<deep_hump and Fr_crest<1
assert np.isclose(hump_roots(0)[1],y_up,rtol=0,atol=1e-10)
assert np.isclose(froude(hump_roots(dz_max)[1],q_hump),1)
try:
    hump_roots(dz_max+.001)
    raise AssertionError("Nemoguće uzvodno stanje nije prepoznato.")
except ValueError:
    pass
print(f"yc={yc_hump:.6f} m, Emin={E_min_hump:.6f} m, Δzmax={dz_max:.6f} m")
print(f"Et={E_up-dz:.6f} m; korijeni {shallow_hump:.6f}/{deep_hump:.6f} m; Frt={Fr_crest:.4f}")
heights = np.linspace(0,dz_max,160)
branches = np.array([hump_roots(h) for h in heights])
fig,ax=plt.subplots(figsize=(7,3.6))
ax.plot(heights,branches[:,1],label='podkritični nastavak')
ax.plot(heights,branches[:,0],ls='--',label='drugi matematički korijen')
ax.axvline(dz_max,color='#c0392b',ls=':',label='kritična granica')
ax.plot(dz,deep_hump,'o',color='#1e8449')
ax.set(xlabel='povišenje dna Δz (m)',ylabel='dubina na tjemenu (m)')
ax.legend();ax.grid(ls=':',alpha=.4);plt.tight_layout();plt.show()

## Z5: zajednički protok u bilanci izmjerenog skoka

Koristimo sintetičke podatke zadatka: \((Q,b,y_1,y_2)=(1{,}800;1{,}200;0{,}250;1{,}220)\)
u SI jedinicama, uz standardne nesigurnosti \((0{,}018;0{,}003;0{,}003;0{,}008)\).
Kontrolni presjeci su izvan valjka. Isti \(q=Q/b\) ulazi u oba člana reziduala.
Prvi postupak koristi centrirane diferencije svih četiriju nezavisnih ulaza.
Za dodatnu Monte Carlo usporedbu pretpostavljamo neovisne normalne raspodjele;
to je dodatna pretpostavka pokusa, a ne posljedica samih oznaka ± u zadatku.

In [ ]:
measured = np.array([1.800,1.200,.250,1.220])
standard_u = np.array([.018,.003,.003,.008])
def jump_residual(data):
    Q,b,y1,y2 = data
    q_measured=Q/b
    return .5*(y2**2-y1**2)+q_measured**2/g*(1/y2-1/y1)
R_measured=jump_residual(measured)
partials=[]
for i in range(4):
    step=np.zeros(4);step[i]=measured[i]*1e-5
    partials.append((jump_residual(measured+step)-jump_residual(measured-step))/(2*step[i]))
u_R_fd=np.linalg.norm(np.array(partials)*standard_u)
rng_z5=np.random.default_rng(1505)
samples=rng_z5.normal(measured[:,None],standard_u[:,None],(4,60000))
R_samples=jump_residual(samples)
u_R_mc=R_samples.std(ddof=1)
assert np.isclose(R_measured,-.0164829974432,rtol=0,atol=1e-10)
assert abs(u_R_mc/u_R_fd-1)<.025
assert abs(R_measured)<2*u_R_fd
different=measured.copy();different[3]=1.1
assert abs(jump_residual(different))>2*u_R_fd
print(f"R={R_measured:.5f} m²; uR={u_R_fd:.5f} m²; |R|/uR={abs(R_measured)/u_R_fd:.3f}")
print(f"Monte Carlo uR={u_R_mc:.5f} m²; zadani kriterij: {abs(R_measured)<=2*u_R_fd}")
fig,ax=plt.subplots(figsize=(7,3.3))
ax.hist(R_samples,bins=60,color='#7cb5d6',edgecolor='white')
ax.axvline(0,color='#1e8449',label='točno zatvorena bilanca')
ax.axvline(R_measured,color='#c0392b',label='srednji mjerni ulazi')
ax.set(xlabel='rezidual M₂ − M₁ (m²)',ylabel='broj uzoraka');ax.legend()
plt.tight_layout();plt.show()

## Z6: održavanje i odvojena provjera bazena

Trapezni kanal ima \(b=3\ \mathrm{m}\), pokos 2:1, konstrukcijsku dubinu
1,50 m i dopuštenu dubinu 1,20 m. Primjenjujemo uniformni Manningov model.
Vrijednost \(n+2u_n\) ovdje je zadani kriterij, bez tvrdnje o zajamčenom intervalu.
Bazen se provjerava zasebno pri propisanoj ulaznoj dubini 0,350 m i širini 5 m.
Veći kapacitet kanala nije automatski stvarni dotok ni dokaz dostatnosti bazena.

In [ ]:
b_trap,z_trap,H,S0,Q_design=3.,2.,1.5,.00150,8.
n_mean=np.array([.018,.026,.035]);n_unc=np.array([.001,.002,.004]);n_cons=n_mean+2*n_unc
y_allow=1.20
def capacity(y,n):
    A=y*(b_trap+z_trap*y);P=b_trap+2*y*np.hypot(1,z_trap)
    return A*(A/P)**(2/3)*np.sqrt(S0)/n
Q_cap=capacity(y_allow,n_mean);Q_cons=capacity(y_allow,n_cons)
depths=np.array([bisect_target(lambda y:capacity(y,n),.01,3.,Q_design)[0] for n in n_mean])
depths_cons=np.array([bisect_target(lambda y:capacity(y,n),.01,3.,Q_design)[0] for n in n_cons])
freeboard=H-depths
y_in_basin,B_basin,y2_allowed=.350,5.,1.40
def basin_y2(flow):
    return conjugate_depth(y_in_basin,flow/(B_basin*y_in_basin))
y2_design=basin_y2(Q_design);y2_cap=basin_y2(Q_cap)
assert np.allclose(capacity(depths,n_mean),Q_design,rtol=0,atol=1e-9)
assert np.all(depths_cons>depths) and np.all(Q_cons<Q_cap)
assert np.array_equal(Q_cons>=Q_design,[True,False,False])
assert np.array_equal(depths_cons<=y_allow,[True,False,False])
assert y2_design<y2_allowed<y2_cap[0]
for i,label in enumerate('ABC'):
    print(f"{label}: Qcap={Q_cap[i]:.3f}, Qc={Q_cons[i]:.3f} m³/s; yn={depths[i]:.3f}, f={freeboard[i]:.3f}, y2(cap)={y2_cap[i]:.3f} m")
print(f"Bazen pri projektnom dotoku: y2={y2_design:.3f} m")
fig,axes=plt.subplots(1,2,figsize=(10,3.7))
ys=np.linspace(.3,1.5,180)
for n,label in zip(n_mean,'ABC'):axes[0].plot(ys,capacity(ys,n),label=label)
axes[0].axhline(Q_design,color='#c0392b',ls='--');axes[0].axvline(y_allow,color='#7a8a96',ls=':')
axes[0].set(xlabel='normalna dubina (m)',ylabel='protok (m³/s)');axes[0].legend()
flows=np.linspace(5,12,160)
axes[1].plot(flows,basin_y2(flows));axes[1].axhline(y2_allowed,color='#c0392b',ls='--',label='dopuštena dubina')
axes[1].plot(Q_cap,y2_cap,'o');axes[1].plot(Q_design,y2_design,'s',color='#1e8449',label='projektni dotok')
axes[1].set(xlabel='dotok bazenu (m³/s)',ylabel='spregnuta dubina (m)');axes[1].legend()
for ax in axes:ax.grid(ls=':',alpha=.4)
plt.tight_layout();plt.show()

## Protumači

1. Zašto dvije jednake specifične energije ne znače da se oba toka mogu ostvariti uz iste rubne uvjete?
2. Kako se mijenja dubina na tjemenu kad prag raste prema graničnoj visini? Zašto drugi korijen ne postaje rješenje bez promjene režima?
3. Što mora promijeniti uzvodno stanje ako je prag viši od graničnog?
4. Zašto se kroz skok koristi bilanca količine gibanja, a mehanička energija pada?
5. Kakvu bi pogrešku dalo neovisno uzorkovanje protoka na dvama presjecima mjernog skoka?
6. Što zaključujemo iz malog reziduala u odnosu na nesigurnost, a što njime još nije dokazano?
7. Zašto čišćenje kanala zahtijeva i provjeru bazena te stvarnih nizvodnih uvjeta?
8. Daje li kriterij \(n+2u_n\) sam po sebi zajamčenu granicu? Koji terenski podatci nedostaju za projektnu odluku?